In [8]:
!pip install roboflow
from google.colab import userdata
from roboflow import Roboflow

# Retrieve the key securely from Colab Secrets
api_key = userdata.get('ROBOFLOW_API_KEY')

# Initialize Roboflow and download the dataset
rf = Roboflow(api_key=api_key)
project = rf.workspace("roboflow-universe-projects").project("construction-site-safety")
version = project.version(30)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.6/333.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Construction-Site-Safety-30 in yolov8:: 100%|██████████| 1446/1446 [00:00<00:00, 4915.39it/s]


Step 1: Downloading the Dataset from Roboflow
What you did: You ran python code using the Roboflow API to download an annotated object detection dataset containing images and bounding boxes for PPE items (Hardhat, Safety Vest, NO-Hardhat, Person, etc.).

Why: Deep learning object detectors like YOLOv8 need thousands of labeled examples to learn what safety gear looks like under different lighting, angles, and distances.

In [13]:
# Cell A: classes and paths
!cat {dataset.location}/data.yaml

names:
- Excavator
- Gloves
- Hardhat
- Ladder
- Mask
- NO-Hardhat
- NO-Mask
- NO-Safety Vest
- Person
- SUV
- Safety Cone
- Safety Vest
- bus
- dump truck
- fire hydrant
- machinery
- mini-van
- sedan
- semi
- trailer
- truck and trailer
- truck
- van
- vehicle
- wheel loader
nc: 25
roboflow:
  license: CC BY 4.0
  project: construction-site-safety
  url: https://universe.roboflow.com/roboflow-universe-projects/construction-site-safety/dataset/30
  version: 30
  workspace: roboflow-universe-projects
test: ../test/images
train: ../train/images
val: ../valid/images


Step 2: Checking data.yaml (Classes & Structure)
What you did (!cat {dataset.location}/data.yaml): You inspected the configuration file created by Roboflow.

Why: data.yaml is the map that tells YOLOv8:

Where to find the images and labels.

How many total classes exist (e.g., nc: 25).

The exact mapping between class numbers (e.g., 0, 1, 2) and names (e.g., 'Hardhat', 'Safety Vest').

In [15]:
print(dataset.location)
!ls {dataset.location}

/content/Construction-Site-Safety-30
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [16]:
!ls {dataset.location}/train
!ls {dataset.location}/train/images | head -5
!ls {dataset.location}/train/images | wc -l

images	labels
000415_jpg.rf.b5634019ce7c1f60117cff1f9daa20c5.jpg
001425_jpg.rf.ccdefc0e04ac537ea9695148b9cc40b3.jpg
002458_jpg.rf.fabab4c3e50cf0357885300322295cf3.jpg
002654_jpg.rf.a9f5806548b78d8f3db1b8423437008f.jpg
002682_jpg.rf.76dcb2ca0de79b894d8a4f12aed4e45f.jpg
521


In [17]:
import os
base = "/content/Construction-Site-Safety-30"
for split in ["train", "valid", "test"]:
    print(split, len(os.listdir(f"{base}/{split}/images")))

train 521
valid 114
test 82


In [19]:
import yaml
base = "/content/Construction-Site-Safety-30"
p = f"{base}/data.yaml"
cfg = yaml.safe_load(open(p))
cfg["train"] = f"{base}/train/images"
cfg["val"]   = f"{base}/valid/images"
cfg["test"]  = f"{base}/test/images"
yaml.safe_dump(cfg, open(p, "w"))
!cat {p}

names:
- Excavator
- Gloves
- Hardhat
- Ladder
- Mask
- NO-Hardhat
- NO-Mask
- NO-Safety Vest
- Person
- SUV
- Safety Cone
- Safety Vest
- bus
- dump truck
- fire hydrant
- machinery
- mini-van
- sedan
- semi
- trailer
- truck and trailer
- truck
- van
- vehicle
- wheel loader
nc: 25
roboflow:
  license: CC BY 4.0
  project: construction-site-safety
  url: https://universe.roboflow.com/roboflow-universe-projects/construction-site-safety/dataset/30
  version: 30
  workspace: roboflow-universe-projects
test: /content/Construction-Site-Safety-30/test/images
train: /content/Construction-Site-Safety-30/train/images
val: /content/Construction-Site-Safety-30/valid/images


Step 3: Checking Image Split Counts
What you did (glob.glob(...) for train, valid, test): You counted how many images are in each split folder.

Why:

Train Set (~70-80%): The images the model learns from.

Validation Set (~10-15%): Used during training to check accuracy at the end of each epoch and detect overfitting.

Test Set (~10-15%): Unseen images used at the very end to evaluate the final model's real-world accuracy.

tep 4: Applying the Absolute Path Fix
What you did: You modified data.yaml using Python's yaml library to rewrite relative paths (e.g., ../train/images) into full absolute paths (e.g., /content/PPE-Detection-30/train/images).

Why: When training on Google Colab or external servers, relative paths frequently break because PyTorch/YOLO might run from a different root directory. Setting absolute paths prevents FileNotFoundError during training.

In [20]:
!pip install ultralytics -q
from ultralytics import YOLO

p = "/content/Construction-Site-Safety-30/data.yaml"
model = YOLO("yolov8n.pt")
model.train(data=p, epochs=5, imgsz=640, batch=16)

Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Construction-Site-Safety-30/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=None, opset=None, optim

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8, 10, 11, 13, 15, 16, 17, 19, 20, 21, 22, 23, 24])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a324431f450>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
p = "/content/Construction-Site-Safety-30/data.yaml"

model = YOLO("yolov8s.pt")
model.train(
    data=p,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,                       # stop early if it stops improving
    classes=[2, 5, 7, 8, 11],          # Hardhat, NO-Hardhat, NO-Safety Vest, Person, Safety Vest
    project="/content/drive/MyDrive/safety-gear-detector/runs",
    name="ppe_yolov8s",
)

In [21]:
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
p = "/content/Construction-Site-Safety-30/data.yaml"

model = YOLO("yolov8s.pt")
model.train(
    data=p,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,                       # stop early if it stops improving
    classes=[2, 5, 7, 8, 11],          # Hardhat, NO-Hardhat, NO-Safety Vest, Person, Safety Vest
    project="/content/drive/MyDrive/safety-gear-detector/runs",
    name="ppe_yolov8s",
)

Mounted at /content/drive
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=[2, 5, 7, 8, 11], close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Construction-Site-Safety-30/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe_

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 2,  5,  7,  8, 11])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a32460d23c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
   

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!ls /content/drive/MyDrive
!ls /content/drive/MyDrive/safety-gear-detector
!find /content/drive/MyDrive -name "best.pt" 2>/dev/null

'12th marksheet .pdf'
 16990895169324823762553607026527.jpg
 AppointmentReciept.pdf
'Colab Notebooks'
 DOC-20260617-WA0000..pdf
 drama.gdoc
 EAadhaar_0013090221169620230421141321_2112202401240.pdf
 GUIDELINES.pdf
 in.gov.abc-ABCID-663049753777.pdf
 in.gov.cbse-SSCER-71992882018.pdf
 Invoice_22299584511.pdf
'Laboratory Equipment.xlsx'
'LEETCODE Problems.gsheet'
'Online Application Receipt (1) (1).pdf'
'Online Application Receipt (1) (2).pdf'
'Online Application Receipt (1).pdf'
'Online Application Receipt.pdf'
 Picsart_24-06-22_01-32-00-539.jpg
'Placement Registration Form 2026-2027.pdf'
 PriyaAkshay23fe10ite00011.pdf
 resume1.pdf
 safety-gear-detector
 Screenshot_2023-11-01-14-08-00-46_e2d5b3f32b79de1d45acd1fad96fbb0f.jpg
 Screenshot_2024-02-18-16-12-29-22_6012fa4d4ddec268fc5c7112cbb265e7.jpg
'Share using'
'small talks and them.gdoc'
 Snapchat-1409565041.jpg
 The%20Bengali%20girl.pdf
'Untitled form.gform'
'Untitled spreadsheet.gsheet'
 weatherforecasting.pdf
runs


In [6]:
!ls -R /content/drive/MyDrive/safety-gear-detector | head -40
!ls /content/drive/MyDrive/runs 2>/dev/null

/content/drive/MyDrive/safety-gear-detector:
runs

/content/drive/MyDrive/safety-gear-detector/runs:
ppe_yolov8s

/content/drive/MyDrive/safety-gear-detector/runs/ppe_yolov8s:
args.yaml
labels.jpg
train_batch0.jpg
train_batch1.jpg
train_batch2.jpg
weights

/content/drive/MyDrive/safety-gear-detector/runs/ppe_yolov8s/weights:


In [9]:
import os
print(os.path.exists("/content/Construction-Site-Safety-30/data.yaml"))

True


In [10]:
!pip install ultralytics -q
from ultralytics import YOLO

p = "/content/Construction-Site-Safety-30/data.yaml"

model = YOLO("yolov8s.pt")
model.train(
    data=p,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=25,
    classes=[2, 5, 7, 8, 11],
    project="/content/runs_local",
    name="ppe_yolov8s_v2",
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=[2, 5, 7, 8, 11], close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Construction-Site-Safety-30/data.ya

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 2,  5,  7,  8, 11])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e7cd0662820>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
   

In [11]:
!ls -la /content/runs_local/ppe_yolov8s_v2/weights
!ls /content/runs_local/ppe_yolov8s_v2


total 44000
drwxr-xr-x 2 root root     4096 Oct  1 23:09 .
drwxr-xr-x 3 root root     4096 Oct  1 23:27 ..
-rw-r--r-- 1 root root 22523818 Oct  1 23:27 best.pt
-rw-r--r-- 1 root root 22523818 Oct  1 23:27 last.pt
args.yaml			 results.csv	      val_batch0_labels.jpg
BoxF1_curve.png			 results.png	      val_batch0_pred.jpg
BoxP_curve.png			 train_batch0.jpg     val_batch1_labels.jpg
BoxPR_curve.png			 train_batch1.jpg     val_batch1_pred.jpg
BoxR_curve.png			 train_batch2970.jpg  val_batch2_labels.jpg
confusion_matrix_normalized.png  train_batch2971.jpg  val_batch2_pred.jpg
confusion_matrix.png		 train_batch2972.jpg  weights
labels.jpg			 train_batch2.jpg


In [12]:
import shutil
shutil.make_archive("/content/ppe_run_v2", "zip", "/content/runs_local/ppe_yolov8s_v2")
from google.colab import files
files.download("/content/ppe_run_v2.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
from google.colab import drive
drive.mount('/content/drive')
import shutil
dst = "/content/drive/MyDrive/safety-gear-detector/runs/ppe_yolov8s_v2"
shutil.copytree("/content/runs_local/ppe_yolov8s_v2", dst, dirs_exist_ok=True)
!ls -la {dst}/weights
drive.flush_and_unmount()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 44000
drwx------ 2 root root     4096 Oct  1 23:09 .
drwx------ 3 root root     4096 Oct  1 23:27 ..
-rw------- 1 root root 22523818 Oct  1 23:27 best.pt
-rw------- 1 root root 22523818 Oct  1 23:27 last.pt


In [14]:
from ultralytics import YOLO
model = YOLO("/content/runs_local/ppe_yolov8s_v2/weights/best.pt")
metrics = model.val(data="/content/Construction-Site-Safety-30/data.yaml",
                    split="test", classes=[2, 5, 7, 8, 11])

Ultralytics 8.4.171 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,135,259 parameters, 0 gradients, 28.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 24.7±11.1 MB/s, size: 130.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/Construction-Site-Safety-30/test/labels... 82 images, 8 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 82/82 333.2it/s 0.2s
val: New cache created: /content/Construction-Site-Safety-30/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.2it/s 5.2s
                   all         82        476      0.848      0.603      0.682      0.447
               Hardhat         30        110          1      0.668      0.799      0.525
            NO-Hardhat         25         41      0.669      0.488      0